# TechGrowth — Simulação de Eventos Discretos (DES)

**Pipeline Kafka → Flink → Iceberg com Injeção de Partições de Rede (PACELC)**

---

| Campo | Valor |
|-------|-------|
| **Disciplina** | MIT-508 — Data Platform Engineering |
| **Instituição** | American Global Tech University (AGTU) |
| **Autor** | Carlos Ulisses Flores |
| **ORCID** | [0000-0002-6034-7765](https://orcid.org/0000-0002-6034-7765) |
| **Repositório** | [github.com/ulissesflores/mit508-techgrowth-des](https://github.com/ulissesflores/mit508-techgrowth-des) |

Este notebook executa a Simulação de Eventos Discretos (DES) estocástica que valida a arquitetura proposta no estudo de caso TechGrowth. A simulação modela o pipeline Kafka → Flink → Iceberg sob burst de Black Friday (5.800 eps) com injeção de partições de rede para validação empírica do teorema PACELC (Abadi, 2012).

## 1. Instalação de Dependências

In [ ]:
!pip install -q simpy numpy matplotlib scipy pandas

## 2. Configuração do Experimento

Parâmetros calibrados via Lei de Little (1961): L = λW.
Para λ_burst = 5.800 eps e W ≈ 23 ms, o dimensionamento de 14 consumers/célula
garante ρ ≈ 0,95 durante burst — estável mas com queuing visível.

In [ ]:
import simpy
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from dataclasses import dataclass
from typing import List, Dict, Tuple
from datetime import datetime
import json
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)

plt.rcParams.update({
    "figure.figsize": (12, 6),
    "figure.dpi": 150,
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


@dataclass
class ExperimentConfig:
    seed: int = 42
    sim_duration_ms: int = 60_000
    nominal_rate_eps: float = 578.0
    burst_rate_eps: float = 5_800.0
    burst_start_ms: int = 15_000
    burst_end_ms: int = 45_000
    num_cells: int = 10
    consumers_per_cell: int = 14
    flink_latency_mean_ms: float = 8.0
    flink_latency_sigma: float = 0.5
    iceberg_commit_ms: float = 15.0
    sla_target_ms: float = 300_000.0
    partition_enabled: bool = True
    partition_probability: float = 0.033
    partition_duration_ms: float = 2_000.0
    partition_latency_multiplier: float = 50.0
    tail_latency_probability: float = 0.003
    tail_latency_multiplier: float = 20.0
    throughput_bin_ms: int = 1_000


@dataclass
class EventRecord:
    event_id: int
    cell_id: int
    arrival_time_ms: float
    processing_start_ms: float
    completion_time_ms: float
    end_to_end_latency_ms: float
    was_during_partition: bool
    was_tail_latency: bool


@dataclass
class PartitionEvent:
    start_ms: float
    end_ms: float
    affected_cell: int


print("Configuração carregada com sucesso.")

## 3. Motor de Simulação DES

A simulação modela:
- **Produtor Kafka** com taxa variável (Poisson): 578 eps nominal → 5.800 eps burst
- **Consumer Group Flink** com 10 células × 14 consumers (SimPy Resources)
- **Sink Iceberg** com latência de commit log-normal
- **Cell-Based Architecture** com shuffle sharding (MacCárthaigh, 2019)
- **Injeção de partições PACELC** com latência multiplicada 50× (Abadi, 2012)
- **Latência de cauda** com probabilidade 0,3% e multiplicador 20× (Dean & Barroso, 2013)

In [ ]:
class TechGrowthPipelineSimulation:
    def __init__(self, config: ExperimentConfig) -> None:
        self.config = config
        self.rng = np.random.RandomState(config.seed)
        self.env = simpy.Environment()
        self.results: List[EventRecord] = []
        self.partition_log: List[PartitionEvent] = []
        self.consumer_lag: List[Tuple[float, int]] = []
        self._pending_events = 0
        self._event_counter = 0
        self.cells = [
            simpy.Resource(self.env, capacity=config.consumers_per_cell)
            for _ in range(config.num_cells)
        ]
        self._cell_partitioned = [False] * config.num_cells

    def _get_arrival_rate(self, current_time_ms: float) -> float:
        if self.config.burst_start_ms <= current_time_ms <= self.config.burst_end_ms:
            return self.config.burst_rate_eps
        return self.config.nominal_rate_eps

    def _assign_cell(self, event_id: int) -> int:
        return event_id % self.config.num_cells

    def _compute_processing_latency(self, cell_id: int) -> float:
        base_latency = self.rng.lognormal(
            mean=np.log(self.config.flink_latency_mean_ms),
            sigma=self.config.flink_latency_sigma,
        )
        iceberg_latency = self.rng.lognormal(
            mean=np.log(self.config.iceberg_commit_ms), sigma=0.3
        )
        total = base_latency + iceberg_latency
        if self._cell_partitioned[cell_id]:
            total *= self.config.partition_latency_multiplier
        if self.rng.random() < self.config.tail_latency_probability:
            total *= self.config.tail_latency_multiplier
        return total

    def _event_producer(self):
        while True:
            rate = self._get_arrival_rate(self.env.now)
            if rate <= 0:
                yield self.env.timeout(1)
                continue
            interval_ms = self.rng.exponential(1000.0 / rate)
            yield self.env.timeout(interval_ms)
            if self.env.now > self.config.sim_duration_ms:
                break
            self._event_counter += 1
            event_id = self._event_counter
            cell_id = self._assign_cell(event_id)
            self.env.process(self._process_event(event_id, cell_id))

    def _process_event(self, event_id: int, cell_id: int):
        arrival_time = self.env.now
        self._pending_events += 1
        self.consumer_lag.append((self.env.now, self._pending_events))
        with self.cells[cell_id].request() as req:
            yield req
            processing_start = self.env.now
            was_partitioned = self._cell_partitioned[cell_id]
            latency = self._compute_processing_latency(cell_id)
            was_tail = latency > (
                self.config.flink_latency_mean_ms + self.config.iceberg_commit_ms
            ) * 10
            yield self.env.timeout(latency)
        completion_time = self.env.now
        self._pending_events -= 1
        self.results.append(EventRecord(
            event_id=event_id, cell_id=cell_id,
            arrival_time_ms=arrival_time,
            processing_start_ms=processing_start,
            completion_time_ms=completion_time,
            end_to_end_latency_ms=completion_time - arrival_time,
            was_during_partition=was_partitioned,
            was_tail_latency=was_tail,
        ))

    def _partition_injector(self):
        if not self.config.partition_enabled:
            return
        while self.env.now < self.config.sim_duration_ms:
            interval = self.rng.exponential(
                1000.0 / self.config.partition_probability
            )
            yield self.env.timeout(interval)
            if self.env.now > self.config.sim_duration_ms:
                break
            affected_cell = self.rng.randint(0, self.config.num_cells)
            self._cell_partitioned[affected_cell] = True
            self.partition_log.append(PartitionEvent(
                start_ms=self.env.now,
                end_ms=self.env.now + self.config.partition_duration_ms,
                affected_cell=affected_cell,
            ))
            yield self.env.timeout(self.config.partition_duration_ms)
            self._cell_partitioned[affected_cell] = False

    def run(self) -> Dict:
        print(f"Iniciando simulação DES (seed={self.config.seed})...")
        self.env.process(self._event_producer())
        self.env.process(self._partition_injector())
        self.env.run(until=self.config.sim_duration_ms)
        return self._compute_metrics()

    def _compute_metrics(self) -> Dict:
        if not self.results:
            return {"error": "Nenhum evento processado"}
        latencies = np.array([r.end_to_end_latency_ms for r in self.results])
        sla = self.config.sla_target_ms
        partition_events = [r for r in self.results if r.was_during_partition]
        partition_latencies = (
            np.array([r.end_to_end_latency_ms for r in partition_events])
            if partition_events else np.array([0.0])
        )
        metrics = {
            "total_events": len(self.results),
            "sla_target_ms": sla,
            "sla_compliance_pct": float(np.mean(latencies <= sla) * 100),
            "p50_latency_ms": float(np.percentile(latencies, 50)),
            "p95_latency_ms": float(np.percentile(latencies, 95)),
            "p99_latency_ms": float(np.percentile(latencies, 99)),
            "max_latency_ms": float(np.max(latencies)),
            "mean_latency_ms": float(np.mean(latencies)),
            "std_latency_ms": float(np.std(latencies)),
            "partitions_injected": len(self.partition_log),
            "events_during_partition": len(partition_events),
            "sla_compliance_during_partition_pct": (
                float(np.mean(partition_latencies <= sla) * 100)
                if partition_events else 100.0
            ),
            "p99_during_partition_ms": (
                float(np.percentile(partition_latencies, 99))
                if len(partition_events) > 1 else 0.0
            ),
            "little_law_L": float(
                np.mean(latencies / 1000.0) * self.config.burst_rate_eps
            ),
        }
        print(f"Eventos processados: {metrics['total_events']:,}")
        print(f"Conformidade SLA: {metrics['sla_compliance_pct']:.1f}%")
        print(f"Latência P50: {metrics['p50_latency_ms']:.1f} ms")
        print(f"Latência P95: {metrics['p95_latency_ms']:.1f} ms")
        print(f"Latência P99: {metrics['p99_latency_ms']:.1f} ms")
        print(f"Partições injetadas: {metrics['partitions_injected']}")
        print(f"SLA durante partição: {metrics['sla_compliance_during_partition_pct']:.1f}%")
        return metrics


print("Motor de simulação carregado.")

## 4. Execução da Simulação

A simulação roda 60 segundos simulados com transição nominal → burst → nominal.

In [ ]:
config = ExperimentConfig()
sim = TechGrowthPipelineSimulation(config)
metrics = sim.run()

## 5. Gráficos de Resultados

Seis figuras em formato de publicação científica com legendas em português.

### Figura 1 — Latência End-to-End ao Longo do Tempo

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
times = [r.arrival_time_ms / 1000 for r in sim.results]
latencies = [r.end_to_end_latency_ms for r in sim.results]
partition_mask = [r.was_during_partition for r in sim.results]

normal_t = [t for t, p in zip(times, partition_mask) if not p]
normal_l = [l for l, p in zip(latencies, partition_mask) if not p]
ax.scatter(normal_t, normal_l, s=1, alpha=0.4, c='#2171b5',
           label='Operação normal', rasterized=True)

part_t = [t for t, p in zip(times, partition_mask) if p]
part_l = [l for l, p in zip(latencies, partition_mask) if p]
if part_t:
    ax.scatter(part_t, part_l, s=3, alpha=0.7, c='#e34a33',
               label='Durante partição PACELC', rasterized=True)

ax.axhline(y=config.sla_target_ms, color='black', linestyle='--',
            linewidth=1, label=f'SLA ({config.sla_target_ms/1000:.0f}s)')
ax.axvspan(config.burst_start_ms/1000, config.burst_end_ms/1000,
           alpha=0.08, color='orange', label='Burst Black Friday (5.800 eps)')
for pe in sim.partition_log:
    ax.axvspan(pe.start_ms/1000, pe.end_ms/1000, alpha=0.15, color='red', linewidth=0)

ax.set_yscale('log')
ax.set_xlabel('Tempo (s)')
ax.set_ylabel('Latência end-to-end (ms)')
ax.set_title('Figura 1 — Latência End-to-End: Operação Normal vs. Partições de Rede (PACELC)')
ax.legend(loc='upper left', framealpha=0.9)
ax.set_ylim(bottom=1)
fig.tight_layout()
plt.show()

### Figura 2 — ECDF de Latência com Marcação de SLA

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
all_lat = sorted([r.end_to_end_latency_ms for r in sim.results])
ecdf_y = np.arange(1, len(all_lat) + 1) / len(all_lat)
ax.plot(all_lat, ecdf_y, color='#2171b5', linewidth=2, label='Todos os eventos')

part_lat = sorted([r.end_to_end_latency_ms for r in sim.results if r.was_during_partition])
if part_lat:
    ecdf_p = np.arange(1, len(part_lat) + 1) / len(part_lat)
    ax.plot(part_lat, ecdf_p, color='#e34a33', linewidth=2, label='Durante partição PACELC')

sla = config.sla_target_ms
ax.axvline(x=sla, color='black', linestyle='--', linewidth=1, label=f'SLA ({sla/1000:.0f}s)')
compliance = np.mean(np.array(all_lat) <= sla) * 100
ax.annotate(f'Conformidade geral: {compliance:.1f}% ≤ SLA',
            xy=(sla * 0.7, 0.5), fontsize=11, fontweight='bold', color='#2171b5')
if part_lat:
    pc = np.mean(np.array(part_lat) <= sla) * 100
    ax.annotate(f'Durante partição: {pc:.1f}% ≤ SLA',
                xy=(sla * 0.7, 0.4), fontsize=11, fontweight='bold', color='#e34a33')

ax.set_xscale('log')
ax.set_xlabel('Latência end-to-end (ms)')
ax.set_ylabel('Probabilidade acumulada')
ax.set_title('Figura 2 — ECDF de Latência: Conformidade com SLA de 5 Minutos')
ax.legend(loc='lower right', framealpha=0.9)
fig.tight_layout()
plt.show()

### Figura 3 — Consumer Lag do Kafka (Lei de Little)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
lag_times = [t / 1000 for t, _ in sim.consumer_lag]
lag_vals = [l for _, l in sim.consumer_lag]
ax.plot(lag_times, lag_vals, color='#2171b5', linewidth=0.5, alpha=0.7)
ax.fill_between(lag_times, lag_vals, alpha=0.2, color='#2171b5')
ax.axvspan(config.burst_start_ms/1000, config.burst_end_ms/1000,
           alpha=0.08, color='orange', label='Burst Black Friday')
little_L = config.burst_rate_eps * np.mean(
    [r.end_to_end_latency_ms / 1000 for r in sim.results])
ax.axhline(y=little_L, color='green', linestyle='--', linewidth=1.5,
           label=f'Lei de Little: L = λW ≈ {little_L:.0f}')
for pe in sim.partition_log:
    ax.axvspan(pe.start_ms/1000, pe.end_ms/1000, alpha=0.15, color='red', linewidth=0)
ax.set_xlabel('Tempo (s)')
ax.set_ylabel('Consumer Lag (eventos pendentes)')
ax.set_title('Figura 3 — Consumer Lag do Kafka: Profundidade de Fila e Lei de Little')
ax.legend(loc='upper left', framealpha=0.9)
fig.tight_layout()
plt.show()

### Figura 4 — Throughput Efetivo

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
bin_ms = config.throughput_bin_ms
bins = np.arange(0, config.sim_duration_ms + bin_ms, bin_ms)
completions = np.array([r.completion_time_ms for r in sim.results])
counts, _ = np.histogram(completions, bins=bins)
throughput = counts / (bin_ms / 1000.0)
bin_centers = (bins[:-1] + bins[1:]) / 2 / 1000
ax.plot(bin_centers, throughput, color='#2171b5', linewidth=1.5, label='Throughput efetivo')
arrival_rate = np.array([
    config.burst_rate_eps if config.burst_start_ms <= t*1000 <= config.burst_end_ms
    else config.nominal_rate_eps for t in bin_centers
])
ax.plot(bin_centers, arrival_rate, color='gray', linestyle='--',
        linewidth=1, alpha=0.7, label='Taxa de chegada')
ax.axvspan(config.burst_start_ms/1000, config.burst_end_ms/1000,
           alpha=0.08, color='orange', label='Burst Black Friday')
ax.set_xlabel('Tempo (s)')
ax.set_ylabel('Eventos por segundo (eps)')
ax.set_title('Figura 4 — Throughput Efetivo: Sustentação sob Burst de Black Friday')
ax.legend(loc='upper right', framealpha=0.9)
fig.tight_layout()
plt.show()

### Figura 5 — Violin Plot por Célula

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
cell_data, cell_labels, affected = [], [], set()
for cid in range(config.num_cells):
    cl = [r.end_to_end_latency_ms for r in sim.results if r.cell_id == cid]
    if cl:
        cell_data.append(cl)
        hp = any(pe.affected_cell == cid for pe in sim.partition_log)
        if hp:
            affected.add(len(cell_data) - 1)
        cell_labels.append(f"Célula {cid}{' ★' if hp else ''}")

parts = ax.violinplot(cell_data, showmeans=True, showmedians=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor('#e34a33' if i in affected else '#2171b5')
    pc.set_alpha(0.6)
ax.set_yscale('log')
ax.set_xticks(range(1, len(cell_labels) + 1))
ax.set_xticklabels(cell_labels, rotation=45, ha='right')
ax.axhline(y=config.sla_target_ms, color='black', linestyle='--',
            linewidth=1, label='SLA (300s)')
ax.set_ylabel('Latência end-to-end (ms) — escala log')
ax.set_title('Figura 5 — Distribuição de Latência por Célula (★ = afetada por partição)')
ax.legend(loc='upper right')
fig.tight_layout()
plt.show()

### Figura 6 — Impacto PACELC

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
normal_lat = [r.end_to_end_latency_ms for r in sim.results if not r.was_during_partition]
partition_lat = [r.end_to_end_latency_ms for r in sim.results if r.was_during_partition]

bp = ax1.boxplot([normal_lat, partition_lat if partition_lat else [0]],
                 tick_labels=['Operação\nNormal', 'Durante\nPartição'],
                 patch_artist=True, showfliers=False)
bp['boxes'][0].set_facecolor('#2171b5'); bp['boxes'][0].set_alpha(0.6)
if partition_lat:
    bp['boxes'][1].set_facecolor('#e34a33'); bp['boxes'][1].set_alpha(0.6)
ax1.set_yscale('log')
ax1.set_ylabel('Latência (ms) — escala log')
ax1.set_title('Comparação de Distribuições')
ax1.axhline(y=config.sla_target_ms, color='black', linestyle='--', linewidth=1, label='SLA')
ax1.legend()

sla = config.sla_target_ms
nc = np.mean(np.array(normal_lat) <= sla) * 100
pc = np.mean(np.array(partition_lat) <= sla) * 100 if partition_lat else 100.0
bars = ax2.bar(['Normal', 'Partição'], [nc, pc], color=['#2171b5', '#e34a33'], alpha=0.7)
ax2.set_ylim(0, 105)
ax2.set_ylabel('Conformidade SLA (%)')
ax2.set_title('Conformidade SLA: Normal vs. Partição')
ax2.axhline(y=99.0, color='green', linestyle='--', linewidth=1, label='Meta 99,0%')
ax2.legend()
for bar, val in zip(bars, [nc, pc]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.1f}%', ha='center', fontweight='bold')
fig.suptitle('Figura 6 — Impacto das Partições de Rede: Validação Empírica do PACELC',
             fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

## 6. Relatório de Resultados

Resumo numérico das métricas obtidas na simulação.

In [ ]:
from IPython.display import Markdown, display

sla_s = config.sla_target_ms / 1000
report_md = f"""
## Resultados da Simulação TechGrowth DES

| Métrica | Valor |
|---------|-------|
| Total de eventos | {metrics['total_events']:,} |
| **Conformidade SLA** | **{metrics['sla_compliance_pct']:.1f}%** |
| Latência P50 | {metrics['p50_latency_ms']:.1f} ms |
| Latência P95 | {metrics['p95_latency_ms']:.1f} ms |
| Latência P99 | {metrics['p99_latency_ms']:.1f} ms |
| Latência máxima | {metrics['max_latency_ms']:.1f} ms |
| Partições injetadas | {metrics['partitions_injected']} |
| Eventos durante partição | {metrics['events_during_partition']} |
| **SLA durante partição** | **{metrics['sla_compliance_during_partition_pct']:.1f}%** |
| Lei de Little (L) | {metrics['little_law_L']:.0f} eventos |

### Conclusão

A arquitetura celular com {config.num_cells} células e {config.consumers_per_cell} consumers/célula
sustenta **{metrics['sla_compliance_pct']:.1f}% de conformidade com o SLA de {sla_s/60:.0f} minutos**
sob burst de Black Friday ({config.burst_rate_eps:.0f} eps). A injeção de
{metrics['partitions_injected']} partições PACELC demonstrou o trade-off PA/EL
com isolamento de blast radius via shuffle sharding.
"""

display(Markdown(report_md))